In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import LwFTrainer, SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils.general import InContextHead, set_seed
from src import models
import copy

### Task Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED, train_val_split_ratio=1)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda", seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

total_param_sum = sum(p.sum().item() for p in model.parameters())
print(total_param_sum)

In [ ]:
trainer = SimpleTrainer(model, seed=None)
trainer.train(train_tasks[0], test_tasks[0], epochs=5, batch_size=64, lr=0.001)
trainer.test(test_tasks)

total_param_sum = sum(p.sum().item() for p in trainer.model.parameters())
print(total_param_sum)

In [ ]:
lwf = LwFTrainer(
    trainer.model,
    paradigm="TIL",
    lmbd = 0.001,
    context_sets=context_sets
)

for i, (train, test) in enumerate(
    zip(train_tasks[1:], test_tasks[1:]), start=1
):
    lwf.train(train, test, batch_size=64, context_id=i, epochs=5, lr=0.001)
    lwf.test(test_tasks, context_list=list(range(5)))

In [4]:
def til_run(config, seed):
    # We need a fresh model for each run
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=seed)
    context_sets = get_context_sets(test_tasks)
    head = InContextHead(context_sets, 10, device="cuda")
    head.set_context(0)
    model = models.get_mnist_model(head=head, device="cuda", seed=seed)
    trainer = SimpleTrainer(model, seed=seed, paradigm='TIL')
    trainer.train(train_tasks[0], val_tasks[0], epochs=5, batch_size=64, lr=0.001, context_id=0)

    # 1. Instantiate the trainer with hyperparameters from the sweep
    lwf = LwFTrainer(
        trainer.model,
        paradigm="TIL",
        lmbd=config.lmbd,
        temp=config.temp,
        context_sets=context_sets
    )

    # 2. Run your training loop
    for i, (train, test) in enumerate(zip(train_tasks[1:], test_tasks[1:]), start=1):
        lwf.train(
            train, 
            test, 
            batch_size=64, 
            context_id=i, 
            epochs=5, 
            lr=0.001,
            weight_decay=config.weight_decay # Use weight_decay from sweep
        )
    
    # 3. Test the final model and log the metric for the sweep
    results = lwf.test(test_tasks, context_list=list(range(len(test_tasks))))
    accuracies = [res[1] for res in results]
    avg_accuracy = sum(accuracies) / len(accuracies)
    return avg_accuracy

def cil_run(config, seed):
    # We need a fresh model for each run
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=seed)
    context_sets = get_context_sets(test_tasks)
    model = models.get_mnist_model(device="cuda", seed=seed)
    trainer = SimpleTrainer(model, seed=seed, paradigm='CIL')
    trainer.train(train_tasks[0], val_tasks[0], epochs=5, batch_size=64, lr=0.001)

    # 1. Instantiate the trainer with hyperparameters from the sweep
    lwf = LwFTrainer(
        trainer.model,
        paradigm="CIL",
        lmbd=config.lmbd,
        temp=config.temp,
        context_sets=context_sets
    )

    # 2. Run your training loop
    for i, (train, test) in enumerate(zip(train_tasks[1:], test_tasks[1:]), start=1):
        lwf.train(
            train, 
            test, 
            batch_size=64, 
            context_id=i, 
            epochs=5, 
            lr=0.001,
            weight_decay=config.weight_decay # Use weight_decay from sweep
        )
    
    # 3. Test the final model and log the metric for the sweep
    results = lwf.test(test_tasks, context_list=list(range(len(test_tasks))))
    accuracies = [res[1] for res in results]
    avg_accuracy = sum(accuracies) / len(accuracies)
    return avg_accuracy

def dil_run(config, seed):
    def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
        """Map the global label to the in context label."""
        return labels % 2
    # We need a fresh model for each run
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=seed)
    context_sets = get_context_sets(test_tasks)
    model = models.get_mnist_model(device="cuda", seed=seed, output_dim=2)
    trainer = SimpleTrainer(model, seed=seed, paradigm='DIL', domain_map_fn=domain_map_fn)
    trainer.train(train_tasks[0], val_tasks[0], epochs=5, batch_size=64, lr=0.001)

    # 1. Instantiate the trainer with hyperparameters from the sweep
    lwf = LwFTrainer(
        trainer.model,
        paradigm="DIL",
        lmbd=config.lmbd,
        temp=config.temp,
        context_sets=context_sets,
        domain_map_fn=domain_map_fn
    )

    # 2. Run your training loop
    for i, (train, test) in enumerate(zip(train_tasks[1:], test_tasks[1:]), start=1):
        lwf.train(
            train, 
            test, 
            batch_size=64, 
            context_id=i, 
            epochs=5, 
            lr=0.001,
            weight_decay=config.weight_decay # Use weight_decay from sweep
        )
    
    # 3. Test the final model and log the metric for the sweep
    results = lwf.test(test_tasks, context_list=list(range(len(test_tasks))))
    accuracies = [res[1] for res in results]
    avg_accuracy = sum(accuracies) / len(accuracies)
    return avg_accuracy

In [ ]:
import wandb

# Assume 'BaseModel', 'LwFTrainer', 'train_tasks', 'test_tasks', 'context_sets'
# are already defined in your script.

sweep_configuration = {
    "method": "bayes",  # Bayesian optimization is efficient for tuning
    "metric": {
        "name": "avg_test_accuracy",
        "goal": "maximize"
    },
    "parameters": {
        "lmbd": {
            "distribution": "log_uniform_values",
            "min": 0.01,
            "max": 1.0
        },
        "temp": {
            "distribution": "uniform",
            "min": 1.0,
            "max": 5.0
        },
        "weight_decay": {
            "distribution": "log_uniform_values",
            "min": 1e-6,
            "max": 1e-2
        }
    }
}

def train_one_run():
    """
    Executes a single training and evaluation run with a set of hyperparameters
    provided by the wandb sweep agent.
    """
    SEED = 0
    with wandb.init(tags="lwf_runs") as run:
        config = wandb.config
        accs = []

        acc = til_run(config, SEED)
        if acc < 0.75:
            wandb.finish(1)
            return
        accs.append(acc)

        acc = dil_run(config, SEED)
        if acc < 0.65:
            wandb.finish(1)
            return
        accs.append(acc)
        
        acc = cil_run(config, SEED)
        accs.append(acc)

        wandb.log({"avg_test_accuracy": sum(accs) / 3})
        wandb.finish()
        

# Initialize the sweep
sweep_id = wandb.sweep(sweep=sweep_configuration, project="LwF-Hyperparam-Sweep")

# Start the agent, which will call `train_one_run` for each set of hyperparameters
# Let's run it for 10 trials as an example
wandb.agent(sweep_id, function=train_one_run, count=100)

Create sweep with ID: kmvab5e2
Sweep URL: https://wandb.ai/agt-team/LwF-Hyperparam-Sweep/sweeps/kmvab5e2


wandb: Agent Starting Run: ur2v61n8 with config:
wandb: 	lmbd: 0.4493241394013499
wandb: 	temp: 3.704996791328249
wandb: 	weight_decay: 0.002237392729788459


Training Epochs: 100%|████████████████████████████████████| 5/5 [00:07<00:00,  1.45s/it, val_loss=3.7915, val_acc=0.0059]


Test Results: [(2.3029, 0.2328), (2.3029, 0.3323), (2.3026, 0.5501), (2.3026, 0.3801), (3.5225, 0.0064)]


wandb: Agent Starting Run: vs8kughw with config:
wandb: 	lmbd: 0.3133286431938244
wandb: 	temp: 1.1583516193137533
wandb: 	weight_decay: 1.0771084739251578e-05


...

wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: 5sa1ddwy with config:
wandb: 	lmbd: 0.2653890408065372
wandb: 	temp: 3.2873544601071822
wandb: 	weight_decay: 2.3957861519518544e-06


Training Epochs: 100%|████████████████████████████████████| 5/5 [00:06<00:00,  1.39s/it, val_loss=2.9441, val_acc=0.0614]


Test Results: [(2.3026, 0.3571), (2.3026, 0.4676), (2.3026, 0.4721), (2.3026, 0.0757), (2.6455, 0.1692)]


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: hvybvb8r with config:
wandb: 	lmbd: 0.022522157540064355
wandb: 	temp: 1.0969098498660408
wandb: 	weight_decay: 4.267121186603569e-06


Training Epochs:  80%|████████████████████████████▊       | 4/5 [00:06<00:01,  1.37s/it, val_loss=0.6679, val_acc=0.9952]